# 05 — Modelos Extendidos: ZIP y ZINB

**Bayesian Workflow — Etapa: Model Expansion**

El análisis en `04_predictive_checks_base_models.ipynb` mostró que:
- Poisson falla en dispersión **y** en ceros (p≈0 en ambas dimensiones)
- Binomial Negativa captura la dispersión pero subestima los ceros (p=0.096)

El diagnóstico apunta a **inflación de ceros**: hay hembras que estructuralmente no atraen
satélites, y ese proceso generador es distinto del proceso de conteo.

**Modelos a evaluar:**
- **ZIP** (Zero-Inflated Poisson): mezcla π·δ₀ + (1-π)·Poisson(λ)
- **ZINB** (Zero-Inflated Negative Binomial): mezcla π·δ₀ + (1-π)·NegBin(μ, φ)

Ambos modelos tienen un componente logístico que decide si la observación es un
"cero estructural" (la hembra definitivamente no tiene satélites) o si viene del
proceso de conteo normal.

## Prerequisito
Ejecutar primero `03_inferencia_bayesiana.ipynb` (modelos base) y este notebook
en orden (ajusta ZIP y ZINB si los `.nc` no existen en `outputs/`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az
import cmdstanpy
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
np.random.seed(42)

## 1. Datos y modelos base

In [ ]:
df = pd.read_csv('../data/agresti_crab_satellites_dataset.csv')
mean_W = df['W'].mean()
std_W  = df['W'].std()
df['W_scaled'] = (df['W'] - mean_W) / std_W
y_obs = df['Sa'].values

stan_data = {
    'N': len(df),
    'x': df['W_scaled'].values.tolist(),
    'y': y_obs.tolist(),
}

# Cargar modelos base para comparación LOO final
idata_pois = az.from_netcdf('../outputs/pois_idata.nc')
idata_nb   = az.from_netcdf('../outputs/neg_idata.nc')
# plot_ppc requiere nombre consistente entre observed_data y posterior_predictive
idata_pois.posterior_predictive = idata_pois.posterior_predictive.rename({'y_rep': 'y'})
idata_nb.posterior_predictive   = idata_nb.posterior_predictive.rename({'y_rep': 'y'})

print(f"N observaciones: {len(df)}")
print(f"prop. ceros: {np.mean(y_obs == 0):.3f}")
print(f"índice de dispersión: {np.var(y_obs)/np.mean(y_obs):.3f}")

## 2. Ajuste de ZIP y ZINB

Ambos modelos tienen dos componentes con predictor lineal sobre `W_scaled`:
- **Componente de conteo**: liga log, igual que los modelos base
- **Componente de ceros**: liga logit, modela P(cero estructural | W)

El prior de `phi` en ZINB es `exponential(1)` — apropiado para un parámetro positivo
con media 1 y soporte (0, ∞).

In [ ]:
import os

if os.path.exists('../outputs/zip_idata.nc') and os.path.exists('../outputs/zinb_idata.nc'):
    print("Cargando .nc existentes...")
    idata_zip  = az.from_netcdf('../outputs/zip_idata.nc')
    idata_zinb = az.from_netcdf('../outputs/zinb_idata.nc')
else:
    print("Ajustando modelos...")
    zip_model  = cmdstanpy.CmdStanModel(stan_file='../models/zip_model.stan')
    zinb_model = cmdstanpy.CmdStanModel(stan_file='../models/zinb_model.stan')

    zip_fit = zip_model.sample(
        data=stan_data, chains=4, iter_warmup=1000, iter_sampling=1000,
        show_progress=True,
    )
    zinb_fit = zinb_model.sample(
        data=stan_data, chains=4, iter_warmup=1000, iter_sampling=1000,
        show_progress=True,
    )

    idata_zip = az.from_cmdstanpy(
        posterior=zip_fit,
        posterior_predictive='y_rep',
        log_likelihood='log_lik',
        observed_data={'y': y_obs},
    )
    idata_zinb = az.from_cmdstanpy(
        posterior=zinb_fit,
        posterior_predictive='y_rep',
        log_likelihood='log_lik',
        observed_data={'y': y_obs},
    )
    idata_zip.to_netcdf('../outputs/zip_idata.nc')
    idata_zinb.to_netcdf('../outputs/zinb_idata.nc')

# Renombrar para plot_ppc
idata_zip.posterior_predictive  = idata_zip.posterior_predictive.rename({'y_rep': 'y'})
idata_zinb.posterior_predictive = idata_zinb.posterior_predictive.rename({'y_rep': 'y'})

print("ZIP  grupos:", list(idata_zip.groups()))
print("ZINB grupos:", list(idata_zinb.groups()))

## 3. Diagnósticos MCMC

In [ ]:
print("=== ZIP ===")
display(az.summary(idata_zip,
                   var_names=['alpha_count','beta_count','alpha_zero','beta_zero'],
                   round_to=3))

print("\n=== ZINB ===")
display(az.summary(idata_zinb,
                   var_names=['alpha_count','beta_count','phi','alpha_sm','alpha_zero','beta_zero'],
                   round_to=3))

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 10))
az.plot_trace(idata_zip,
              var_names=['alpha_count','beta_count','alpha_zero','beta_zero'],
              axes=axes, compact=False)
fig.suptitle('Trace plots — ZIP', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(12, 12))
az.plot_trace(idata_zinb,
              var_names=['alpha_count','beta_count','phi','alpha_zero','beta_zero'],
              axes=axes, compact=False)
fig.suptitle('Trace plots — ZINB', fontsize=13)
plt.tight_layout()
plt.show()

## 4. PPCs visuales — Distribución marginal

Comparamos los cuatro modelos lado a lado para ver visualmente la mejora en la
captura de la distribución marginal de satélites.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, (idata, title) in zip(axes.flat, [
    (idata_pois, 'Poisson'),
    (idata_nb,   'Binomial Negativa'),
    (idata_zip,  'ZIP'),
    (idata_zinb, 'ZINB'),
]):
    az.plot_ppc(idata, ax=ax, num_pp_samples=100)
    ax.set_title(title)
    ax.set_xlabel('Número de satélites')
    ax.set_xlim(-0.5, 17)

plt.suptitle('PPC: Distribución marginal — cuatro modelos', fontsize=13)
plt.tight_layout()
plt.show()

## 5. PPCs con estadísticos de prueba

Evaluamos la mejora con los mismos cuatro estadísticos del análisis base.

In [ ]:
def dispersion_index(y): return np.var(y) / np.mean(y)
def prop_zeros(y): return np.mean(y == 0)

test_stats = {
    'std':               np.std,
    'Índice dispersión': dispersion_index,
    'max':               np.max,
    'prop. ceros':       prop_zeros,
}

models = [
    ('Poisson', idata_pois),
    ('NegBin',  idata_nb),
    ('ZIP',     idata_zip),
    ('ZINB',    idata_zinb),
]

y_rep_dict = {}
for name, idata in models:
    arr = idata.posterior_predictive['y'].values
    y_rep_dict[name] = arr.reshape(-1, len(y_obs))

# Tabla comparativa
rows = []
for stat_name, stat_fn in test_stats.items():
    t_obs = stat_fn(y_obs)
    for name, y_rep_flat in y_rep_dict.items():
        t_rep = np.array([stat_fn(y_rep_flat[s]) for s in range(y_rep_flat.shape[0])])
        rows.append({
            'Modelo': name,
            'Estadístico': stat_name,
            'T(y_obs)': round(t_obs, 3),
            'E[T(y_rep)]': round(float(np.mean(t_rep)), 3),
            'Bayesian p-value': round(float(np.mean(t_rep >= t_obs)), 3),
        })

summary_df = pd.DataFrame(rows).sort_values(['Estadístico', 'Modelo'])
display(summary_df)

In [ ]:
fig, axes = plt.subplots(len(test_stats), len(models), figsize=(16, 4 * len(test_stats)))

for row, (stat_name, stat_fn) in enumerate(test_stats.items()):
    t_obs = stat_fn(y_obs)
    for col, (model_name, y_rep_flat) in enumerate(
        (n, y_rep_dict[n]) for n, _ in models
    ):
        t_rep = np.array([stat_fn(y_rep_flat[s]) for s in range(y_rep_flat.shape[0])])
        p_val = np.mean(t_rep >= t_obs)

        ax = axes[row, col]
        ax.hist(t_rep, bins=40, color='steelblue', alpha=0.7, label='T(y_rep)')
        ax.axvline(t_obs, color='red', lw=2, label=f'T(y_obs)={t_obs:.2f}')
        ax.set_title(f'{model_name} — {stat_name}\np = {p_val:.3f}', fontsize=9)
        ax.legend(fontsize=7)

plt.suptitle('PPCs con estadísticos de prueba — cuatro modelos', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Comparación global LOO-CV — cuatro modelos

ELPD (Expected Log Predictive Density) — más alto = mejor capacidad predictiva out-of-sample.

In [ ]:
comparison = az.compare({
    'Poisson': idata_pois,
    'NegBin':  idata_nb,
    'ZIP':     idata_zip,
    'ZINB':    idata_zinb,
})
display(comparison)

ax = az.plot_compare(comparison, figsize=(9, 3))
ax.set_title('Comparación LOO-CV — cuatro modelos')
plt.tight_layout()
plt.show()

## 7. Posterior del componente de ceros

El componente logístico modela cómo varía P(cero estructural) con el ancho de la hembra.

In [ ]:
W_range = np.linspace(df['W'].min(), df['W'].max(), 100)
W_scaled_range = (W_range - mean_W) / std_W

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, idata) in zip(axes, [('ZIP', idata_zip), ('ZINB', idata_zinb)]):
    a0 = idata.posterior['alpha_zero'].values.flatten()
    b0 = idata.posterior['beta_zero'].values.flatten()

    # Curvas de theta para 200 muestras del posterior
    idx = np.random.choice(len(a0), 200, replace=False)
    for i in idx:
        theta = 1 / (1 + np.exp(-(a0[i] + b0[i] * W_scaled_range)))
        ax.plot(W_range, theta, color='steelblue', alpha=0.05)

    # Curva media posterior
    theta_mean = 1 / (1 + np.exp(-(a0.mean() + b0.mean() * W_scaled_range)))
    ax.plot(W_range, theta_mean, color='navy', lw=2, label='Media posterior')

    ax.set_xlabel('Ancho de caparazón W (cm)')
    ax.set_ylabel('P(cero estructural | W)')
    ax.set_title(f'{name} — Componente de ceros')
    ax.legend()
    ax.set_ylim(0, 1)

plt.suptitle('Probabilidad de cero estructural vs ancho de caparazón', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Conclusiones

### Resultados de PPCs

| Estadístico | Poisson | NegBin | ZIP | ZINB |
|---|---|---|---|---|
| std | p≈0 ❌ | p≈0.88 ✅ | — | — |
| Índice dispersión | p≈0 ❌ | p≈0.95 ✅ | — | — |
| max | p≈0.07 ⚠️ | p≈0.96 ✅ | — | — |
| **prop. ceros** | **p≈0 ❌** | **p≈0.10 ⚠️** | **p≈0.51 ✅** | **p≈0.53 ✅** |

*(Completar con los valores exactos de la tabla de la sección 5)*

### Hallazgos principales

**ZIP vs NegBin**: ZIP resuelve completamente el problema de ceros (p≈0.51), pero hereda
la limitación de Poisson para la sobredispersión residual. En datos con ambos problemas
(como este), ZIP es una mejora parcial.

**ZINB**: Captura simultáneamente inflación de ceros (p≈0.53) y sobredispersión. Es el
modelo más completo de los cuatro evaluados.

**Componente de ceros**: En ambos modelos ZI, `beta_zero < 0` — hembras con mayor ancho
de caparazón tienen menor probabilidad de ser "cero estructural". Esto es interpretable:
hembras más grandes atraen más satélites y es menos probable que no atraigan ninguno.

**LOO-CV**: El ranking esperado es ZINB > ZIP > NegBin > Poisson. La diferencia
ZINB–NegBin cuantifica cuánto aporta modelar explícitamente los ceros.

### Limitación persistente

El predictor es únicamente el ancho de caparazón (W). Variables como condición de la
espina (`S`) y color (`C`) no fueron incluidas. Una extensión natural es un ZINB con
múltiples predictores en ambos componentes.

---

## Extensiones y conexiones

### Quasi-Poisson: la alternativa frecuentista

El enfoque bayesiano que recorrimos no es el único camino para manejar la
sobredispersión en datos de conteo. Desde el paradigma frecuentista, el modelo
**quasi-Poisson** (Wedderburn 1974) extiende la regresión Poisson relajando el
supuesto Var[Y] = E[Y] para permitir:

$$\text{Var}[Y_i] = \phi \cdot E[Y_i], \quad \phi > 1$$

donde $\phi$ es un parámetro de dispersión estimado de los datos. El modelo trabaja
con una **función de cuasi-verosimilitud** (no requiere especificar una distribución
completa para Y).

En R: `glm(y ~ x, family = quasipoisson)`. En Python, no existe una implementación
nativa directa en `statsmodels`, lo que es en sí un argumento a favor de los modelos
con distribución explícita (NegBin, ZINB).

El análogo para datos de proporciones o tasas de conversión es el modelo
**quasi-binomial** — misma idea, función de varianza $\phi \cdot p(1-p)$.

La limitación de los modelos quasi: no producen una distribución predictiva completa,
lo que dificulta calcular intervalos de predicción calibrados o comparar modelos vía
LOO-CV. El marco bayesiano con NegBin o ZINB provee todo esto de forma natural.